# Installing Starter Code


In [2]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.2 minutes


In [4]:
import pandas as pd

In [5]:
print(ZONES)
print(TIME_BLOCKS)
print(COSTS)

print((times > 45).sum())
print((times > 45).mean())

['Central', 'North', 'Far West']
['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
81
0.38571428571428573


In [6]:
# Part 1. a)

def late_orders(zone, time_block, promise, seed=1):

  if zone == "all":
    zone = ZONES
  else:
    zone = [zone]

  if time_block == "all":
    time_block = TIME_BLOCKS
  else:
    time_block = [time_block]

  all_times = []

  # Generate delivery times for each zone and time block
  for i in zone:
    for j in time_block:
      times = delivery_times(i, j, promise, seed=seed)
      all_times.append(times)

  times = np.concatenate(all_times)
  result = ((times > promise).mean()*100)
  return(result)


print(late_orders("Far West", "Fri/Sat eve", 45), "%")
print(late_orders("all", "Lunch", 45), "%")
print(late_orders("Far West", "all", 45), "%")
print(late_orders("all", "all", 45), "%")

print(f"{late_orders('Far West', 'Fri/Sat eve', 45):.1f}%")

36.666666666666664 %
3.3333333333333335 %
15.606936416184972 %
8.291873963515755 %
36.7%


In [7]:
# Part 1. b)

rankings = []
for i in ZONES:
  for j in TIME_BLOCKS:
    rate = late_orders(i, j, 45)
    rankings.append((i, j, rate))

rankings = sorted(rankings, key=lambda x: x[2], reverse=True)

for i, j, rate in rankings:
  print(f"{i}, {j}: {rate:.1f}%")

Far West, Fri/Sat eve: 36.7%
North, Fri/Sat eve: 16.7%
Far West, Weekday eve: 11.2%
Central, Fri/Sat eve: 6.8%
North, Weekday eve: 6.0%
Far West, Other: 4.7%
Central, Weekday eve: 4.2%
North, Lunch: 3.7%
Central, Lunch: 3.5%
Central, Other: 2.6%
North, Other: 2.5%
Far West, Lunch: 2.5%


In [8]:
# Part 1. c).

def average_delivery_time(zone, time_block, promise, seed=1):

  if zone == "all":
    zone = ZONES
  else:
    zone = [zone]

  if time_block == "all":
    time_block = TIME_BLOCKS
  else:
    time_block = [time_block]

  all_times = []

  # Generate delivery times for each zone and time block
  for i in zone:
    for j in time_block:
      times = delivery_times(i, j, promise, seed=seed)
      all_times.append(times)

  times = np.concatenate(all_times)
  result = times.mean()

  return result

  # testing
print(f"{average_delivery_time('Far West', 'Fri/Sat eve', 45):.1f} minutes")
print(f"{average_delivery_time('all', 'Lunch', 45):.1f} minutes")
print(f"{average_delivery_time('Far West', 'all', 45):.1f} minutes")
print(f"{average_delivery_time('all', 'all', 45):.1f} minutes")

42.9 minutes
28.2 minutes
35.3 minutes
31.2 minutes


In [9]:
# Part 1. d)

rankings = []

for i in ZONES:
  for j in TIME_BLOCKS:
    rate = average_delivery_time(i, j, 45)
    rankings.append((i, j, rate))

# Sort from longest to shortest average delivery time
rankings = sorted(rankings, key=lambda x: x[2], reverse=True)

for i, j, rate in rankings:
  print(f"{i}, {j}: {rate:.1f} minutes")


Far West, Fri/Sat eve: 42.9 minutes
North, Fri/Sat eve: 36.8 minutes
Far West, Weekday eve: 35.0 minutes
Central, Fri/Sat eve: 32.1 minutes
North, Weekday eve: 31.4 minutes
Far West, Lunch: 30.6 minutes
Far West, Other: 30.0 minutes
Central, Weekday eve: 29.0 minutes
North, Lunch: 28.7 minutes
North, Other: 27.7 minutes
Central, Lunch: 26.8 minutes
Central, Other: 26.0 minutes


# Part 1. e)
The late rate is more relevant to Rosa's decisions because it shows how often orders are actually late compared to the 45-minute promise. In Part B, the highest late rate was Far West, Fri/Sat eve at 36.7%, which means more than one-third of the orders were late. The next highest was North, Fri/Sat eve at 16.7%. The average delivery time in Part D also shows how Far West, Fri/Sat eve is the biggest problem, with an average delivery time of 42.9 minutes. North, Fri/Sat eve was next at 36.8 minutes. However, average delivery time does not directly show how many orders were late. For example, an average could be below 45 minutes even if some orders were still late. Since Rosa's promise is 45 minutes, I think the late rate is more useful for deciding where there is a problem. The average delivery time is still useful because it shows which areas and time blocks generally take longer, but the late rate gives a clearer idea of whether Rosa is meeting her promise to customers.

In [10]:
# Part 2. a)

def late_order_cost(costs):
  refund = costs["refund"]
  churn = costs["churn_orders"]
  margin = costs["margin"]

  # Total cost of a late order = refund + lost profit from churn
  total_cost = refund + (churn * margin)

  return total_cost

print("Cost per late order:", late_order_cost(COSTS))

Cost per late order: 26.2


In [11]:
# Part 2. b)

def best_promise(zone, time_block, promises, costs):

  best_promise = None
  best_profit = None

  # Calculate the cost of each late order
  late_cost = late_order_cost(costs)

  # Test each promised delivery time
  for promise in promises:
    times = delivery_times(zone, time_block, promise, seed=1)

    orders = len(times)
    late = (times > promise).sum()

    profit = orders * costs["margin"]
    late_costs = late * late_cost


    # Net profit = profit from orders - cost of late orders
    net_profit = profit - late_costs

    if best_profit is None or net_profit > best_profit:
      best_profit = net_profit
      best_promise = promise

  return best_promise, best_profit

# Test promises
promises = range(25, 75, 5)

result = best_promise(
  "Far West",
  "Fri/Sat eve",
  promises,
  COSTS
)

print("Best promise:", result[0], "minutes")
print("Net profit:", result[1])

Best promise: 55 minutes
Net profit: 1212.4


# Part 2. b).

I first tested promised delivery times from 5 to 50 minutes in 5-minute steps. The best promise was 50 minutes, but since 50 was the largest value I tested, I thought the best promise could be higher. When the best result is at the edge of the range, it makes sense to increase the range and test longer promised times.

When I increased the range, I found that the longer promises gave better results, so I changed the range to 25 to 75 minutes in 5-minute steps. I chose this range because promises that are too short can result in a large number of late orders and higher costs, while promises that are too long can reduce the number of customers who order. Testing from 25 to 75 minutes gives a wider range of realistic promised delivery times and makes it less likely that the best promise will be at the edge of the range.

My final range is:

promises = range(25, 76, 5)

I then used this range to compare the net profit for each promised delivery time and choose the promise with the highest net profit.

In [14]:
# Net profit for each promise, to check the range is wide enough
# This was strongly suggested by Claude because when you run it,
# you'll see that profit is very negative at short promises, peaks at 55,
# and then falls.

late_cost = late_order_cost(COSTS)

for promise in range(5, 81, 5):
  times = delivery_times("Far West", "Fri/Sat eve", promise, seed=1)
  orders = len(times)
  late = (times > promise).sum()
  net_profit = orders * COSTS["margin"] - late * late_cost
  print(f"{promise} min: {orders} orders, {late} late, net profit {net_profit:.1f}")

5 min: 242 orders, 242 late, net profit -4162.4
10 min: 242 orders, 242 late, net profit -4162.4
15 min: 241 orders, 241 late, net profit -4145.2
20 min: 241 orders, 239 late, net profit -4092.8
25 min: 239 orders, 233 late, net profit -3953.6
30 min: 237 orders, 218 late, net profit -3578.6
35 min: 232 orders, 179 late, net profit -2601.8
40 min: 224 orders, 133 late, net profit -1468.6
45 min: 210 orders, 77 late, net profit -127.4
50 min: 188 orders, 33 late, net profit 827.4
55 min: 158 orders, 8 late, net profit 1212.4
60 min: 121 orders, 2 late, net profit 1036.6
65 min: 84 orders, 0 late, net profit 756.0
70 min: 54 orders, 0 late, net profit 486.0
75 min: 32 orders, 0 late, net profit 288.0
80 min: 18 orders, 0 late, net profit 162.0
